# Wikipedia Language Analytics

**Quantitative Analysis of Language and Sentiment Patterns in Wikipedia Articles**

This notebook combines NLP, graph analytics, and statistics to study *how* Wikipedia describes different topics — not to prove bias, but to let the numbers speak.

| Section | What it does |
|---|---|
| 1 | Single-article sentiment progression |
| 2 | Emotion profile (7-class) |
| 3 | Section-wise sentiment |
| 4 | Language metrics (readability, lexical diversity, term densities) |
| 5 | Entity-level sentiment via spaCy NER |
| 6 | Comparative group analysis |
| 7 | Statistical testing (Welch t, Mann-Whitney U) |
| 8 | Sentiment knowledge graph + centrality |
| 9 | Temporal analysis (revision history) |
| 10 | Word cloud |

**Recommended runtime:** GPU (Runtime → Change runtime type → T4 GPU)

In [ ]:
# ── Colab setup — run once ────────────────────────────────────────────────────
import os

if not os.path.exists('src'):
    !git clone https://github.com/RazaTypesCode/sentiment-analysis-wikipedia.git
    %cd sentiment-analysis-wikipedia

!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q

In [ ]:
import sys
sys.path.insert(0, '.')

import matplotlib.pyplot as plt
import pandas as pd

from src.wiki_loader import (
    get_article, get_sections, build_graph_data,
    count_references, get_revision_ids, get_article_by_revision,
)
from src.sentiment import (
    analyze, analyze_sections, average_score, section_averages,
    analyze_emotions, emotion_profile,
)
from src.text_metrics import compute_all_metrics, compute_group_metrics
from src.entity_sentiment import analyze_top_entities
from src.graph_analytics import build_digraph, centrality_sentiment_table, high_sentiment_hubs
from src.statistics import compare_groups, print_comparison, compare_many
from src.article_sets import SETS
from src.visualization import (
    plot_sentiment_progression, plot_section_sentiment,
    plot_article_comparison, plot_group_boxplot,
    plot_entity_sentiment, plot_emotion_bars, plot_emotion_heatmap,
    plot_radar, plot_knowledge_graph, plot_centrality_scatter,
    plot_temporal_sentiment, plot_wordcloud,
)

print('All imports OK')

## 1. Single Article — Sentiment Progression

The article text is split into 400-character chunks. RoBERTa (trained on diverse English corpora — news, reviews, social media) classifies each chunk as POSITIVE or NEGATIVE. Scores are mapped to `[-1, +1]` and plotted in reading order.

In [ ]:
ARTICLE = 'Quantum mechanics'   # ← change to any Wikipedia article

print(f'Fetching: {ARTICLE}')
text = get_article(ARTICLE)
refs = count_references(ARTICLE)
print(f'Characters : {len(text):,}')
print(f'References : {refs}')

print('\nRunning sentiment analysis (RoBERTa)...')
results = analyze(text)
avg = average_score(results)
print(f'Chunks     : {len(results)}')
print(f'Avg score  : {avg:+.3f}  ({"positive" if avg > 0 else "negative"})')

In [ ]:
fig = plot_sentiment_progression(results, title=f'Sentiment Progression — {ARTICLE}')
plt.show()

## 2. Emotion Profile

A DistilRoBERTa model fine-tuned on 7 emotion classes (anger, disgust, fear, joy, neutral, sadness, surprise) scores the same text. Average scores across all chunks reveal which emotions dominate the article's language.

In [ ]:
print('Running emotion analysis...')
emotions = analyze_emotions(text)

for emotion, score in sorted(emotions.items(), key=lambda x: -x[1]):
    bar = '█' * int(score * 30)
    print(f'  {emotion:<10}  {score:.3f}  {bar}')

fig = plot_emotion_bars(emotions, title=f'Emotion Profile — {ARTICLE}')
plt.show()

## 3. Section-Wise Sentiment

Each named section of the article is analyzed independently. This shows which parts of the article carry the most positive or negative language.

In [ ]:
sections = get_sections(ARTICLE)
print(f'Found {len(sections)} sections')

sec_results = analyze_sections(sections)
sec_avgs = section_averages(sec_results)

fig = plot_section_sentiment(sec_avgs, title=f'Section Sentiment — {ARTICLE}')
plt.show()

## 4. Language Metrics

Multi-dimensional language analysis beyond sentiment:

| Metric | Description |
|---|---|
| avg_sentence_length | Mean words per sentence |
| lexical_diversity | Unique words ÷ total words (type-token ratio) |
| flesch_reading_ease | Higher = easier to read (0–100 scale) |
| flesch_kincaid_grade | US grade level required to read |
| violence_density | Violence-related words per 1 000 |
| uncertainty_density | Hedging / epistemic words per 1 000 |
| emotional_density | Evaluative / emotional words per 1 000 |

In [ ]:
metrics = compute_all_metrics(text)
print(f'Language metrics for: {ARTICLE}\n')
for k, v in metrics.items():
    print(f'  {k:<25}  {v}')

In [ ]:
# Compare metrics across a set of articles
COMPARE_ARTICLES = [
    'Quantum mechanics',
    'Climate change',
    'Nigerian Civil War',
    'English Civil War',
    'Nelson Mandela',
    'Winston Churchill',
]

texts = {title: get_article(title) for title in COMPARE_ARTICLES}
all_metrics = compute_group_metrics(texts)

metrics_df = pd.DataFrame(all_metrics).T
display(metrics_df.style.background_gradient(cmap='RdYlGn', axis=0))

In [ ]:
# Radar chart — pick two articles to compare
A_TITLE = 'Nigerian Civil War'
B_TITLE = 'English Civil War'

RADAR_METRICS = ['avg_sentence_length', 'lexical_diversity',
                 'violence_density', 'uncertainty_density', 'emotional_density']

radar_data = {
    A_TITLE: {m: all_metrics[A_TITLE][m] for m in RADAR_METRICS},
    B_TITLE: {m: all_metrics[B_TITLE][m] for m in RADAR_METRICS},
}

fig = plot_radar(radar_data, title=f'Language Metrics Radar: {A_TITLE} vs {B_TITLE}')
plt.show()

## 5. Entity-Level Sentiment

spaCy extracts the most frequent named entities (persons, organisations, places, events). For each entity, all surrounding context windows (±300 chars per mention) are collected and scored. The result is a per-entity sentiment bar chart.

In [ ]:
ENTITY_ARTICLE = 'Nelson Mandela'   # ← works well for this section
print(f'Fetching: {ENTITY_ARTICLE}')
entity_text = get_article(ENTITY_ARTICLE)

print('Extracting entities and scoring contexts...')
entity_scores = analyze_top_entities(entity_text, top_n=12)

print('\nEntity sentiment scores:')
for ent, score in sorted(entity_scores.items(), key=lambda x: -x[1]):
    sign = '+' if score >= 0 else ''
    print(f'  {sign}{score:.3f}  {ent}')

fig = plot_entity_sentiment(entity_scores, title=f'Entity Sentiment — {ENTITY_ARTICLE}')
plt.show()

## 6. Comparative Group Analysis

Analyze all articles in two named groups from `src/article_sets.py`. Compare sentiment distributions, language metrics, and emotions side-by-side.

Available groups: see `SETS.keys()` in the next cell.

In [ ]:
print('Available article sets:')
for name, articles in SETS.items():
    print(f'  {name}  ({len(articles)} articles)')

In [ ]:
GROUP_A_NAME = 'Civil Wars — Global South'
GROUP_B_NAME = 'Civil Wars — Global North'

group_a_titles = SETS[GROUP_A_NAME]
group_b_titles = SETS[GROUP_B_NAME]

def fetch_and_score(titles):
    scores = {}
    texts  = {}
    for title in titles:
        print(f'  Fetching: {title}')
        try:
            t = get_article(title)
            texts[title]  = t
            scores[title] = average_score(analyze(t))
            print(f'    → {scores[title]:+.3f}')
        except Exception as e:
            print(f'    skipped: {e}')
    return scores, texts

print(f'\n{GROUP_A_NAME}')
scores_a, texts_a = fetch_and_score(group_a_titles)

print(f'\n{GROUP_B_NAME}')
scores_b, texts_b = fetch_and_score(group_b_titles)

In [ ]:
fig = plot_group_boxplot(
    {GROUP_A_NAME: list(scores_a.values()),
     GROUP_B_NAME: list(scores_b.values())},
    title='Sentiment Distribution by Group',
)
plt.show()

In [ ]:
# Emotion heatmap across all articles in both groups
all_texts = {**texts_a, **texts_b}
profiles = emotion_profile(all_texts)

fig = plot_emotion_heatmap(profiles, title='Emotion Heatmap — Both Groups')
plt.show()

## 7. Statistical Testing

Two tests are run on the sentiment scores of each group pair:

- **Welch's t-test** — does not assume equal variance; appropriate for unequal group sizes.
- **Mann-Whitney U** — non-parametric; robust when samples are small or non-normal.

Both use a two-sided alternative (`H₁: means differ`). A result is marked significant if `p < 0.05` on either test.

In [ ]:
result = compare_groups(
    list(scores_a.values()), list(scores_b.values()),
    name_a=GROUP_A_NAME, name_b=GROUP_B_NAME,
)
print_comparison(result)

In [ ]:
# Compare all four leader groups pairwise
MULTI_GROUPS = {
    'Leaders — Africa' : SETS['Leaders — Africa'],
    'Leaders — Europe' : SETS['Leaders — Europe'],
    'Scientists — Western' : SETS['Scientists — Western'],
    'Scientists — Non-Western': SETS['Scientists — Non-Western'],
}

multi_scores = {}
for name, titles in MULTI_GROUPS.items():
    print(f'Fetching: {name}')
    group_scores = []
    for title in titles:
        try:
            t = get_article(title)
            group_scores.append(average_score(analyze(t)))
        except Exception:
            pass
    multi_scores[name] = group_scores
    print(f'  n={len(group_scores)}  mean={sum(group_scores)/len(group_scores):+.3f}')

fig = plot_group_boxplot(multi_scores, title='Sentiment by Article Group')
plt.show()

pairwise = compare_many(multi_scores)
for (a, b), res in pairwise.items():
    sig = '★' if res['significant_05'] else ' '
    pa  = res['welch_t']['p_value']
    pb  = res['mann_whitney_u']['p_value']
    da  = res['group_a']['mean']
    db  = res['group_b']['mean']
    print(f'{sig} {a:30s} vs {b:30s}  Δmean={da-db:+.3f}  t-p={pa:.3f}  U-p={pb:.3f}')

## 8. Sentiment Knowledge Graph + Centrality

BFS from a seed article along Wikipedia hyperlinks. Every node is scored for sentiment. The graph is plotted with:
- **Node color** → red (negative) to green (positive)
- **Node size** → magnitude of sentiment

PageRank and betweenness centrality are then correlated with sentiment to identify high-influence nodes.

In [ ]:
SEED      = 'Quantum mechanics'
DEPTH     = 1
MAX_LINKS = 8

print(f'Building graph from "{SEED}" (depth={DEPTH}, max_links={MAX_LINKS})')
nodes, edges = build_graph_data(SEED, depth=DEPTH, max_links=MAX_LINKS)
print(f'Nodes: {len(nodes)}  Edges: {len(edges)}')

In [ ]:
node_scores = {}
for i, title in enumerate(nodes):
    print(f'[{i+1}/{len(nodes)}] {title}')
    try:
        t = get_article(title)
        node_scores[title] = average_score(analyze(t))
        print(f'    {node_scores[title]:+.3f}')
    except Exception as exc:
        print(f'    skipped: {exc}')
        node_scores[title] = 0.0

In [ ]:
fig = plot_knowledge_graph(nodes, edges, node_scores,
                           title=f'Sentiment Knowledge Graph — {SEED}')
plt.show()

In [ ]:
G = build_digraph(nodes, edges)
df_centrality = centrality_sentiment_table(G, node_scores)
display(df_centrality.head(10))

print('\nHigh-sentiment hubs (high PageRank × high |sentiment|):')
display(high_sentiment_hubs(df_centrality))

fig = plot_centrality_scatter(df_centrality,
                              title=f'PageRank vs Sentiment — {SEED} Graph')
plt.show()

## 9. Temporal Analysis — How Has Language Changed?

Wikipedia exposes its full revision history via the MediaWiki API. We fetch the 8 most recent revisions of an article, score each one, and plot how average sentiment has shifted over time.

> This is an approximation: revision frequency varies widely, and markup stripping may introduce noise.

In [ ]:
TEMPORAL_ARTICLE = 'Climate change'
N_REVISIONS = 8

print(f'Fetching revision history for: {TEMPORAL_ARTICLE}')
revisions = get_revision_ids(TEMPORAL_ARTICLE, limit=N_REVISIONS)
print(f'Found {len(revisions)} revisions')

temporal_scores = []
timestamps = []

for rev in revisions:
    revid = rev['revid']
    ts    = rev['timestamp']
    print(f'  Scoring revision {revid} ({ts[:10]})')
    try:
        rev_text = get_article_by_revision(revid)
        score = average_score(analyze(rev_text))
        temporal_scores.append(score)
        timestamps.append(ts)
        print(f'    {score:+.3f}')
    except Exception as exc:
        print(f'    skipped: {exc}')

# Plot oldest → newest
fig = plot_temporal_sentiment(
    timestamps[::-1], temporal_scores[::-1],
    title=f'Sentiment Over Time — {TEMPORAL_ARTICLE}',
)
plt.show()

## 10. Word Cloud

In [ ]:
fig = plot_wordcloud(text, title=f'Word Cloud — {ARTICLE}')
plt.show()